In [1]:
import torch
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/oritdb_pairs.csv")

In [3]:
filtered_df = df.copy()

# get rid of oriT samples with no matching relaxase sequence or specified nic site
filtered_df = filtered_df[filtered_df["relaxase_sequence"].notna() & (df["nic site"] != '_')]

filtered_df = filtered_df[filtered_df["oriTDB_id"].notna()]

# only keep relaxase sequences larger than 100 
filtered_df = filtered_df[filtered_df["relaxase_sequence"].str.len() > 100]


#filtered_df = filtered_df.drop_duplicates(subset=["oriTDB_id", "oriT_nic_seq"], keep='first')

In [4]:
import torch
from esm.models.esmc import EsmcForMaskedLM, EsmcTokenizer
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ESMC-300M Pretrained model
model = EsmcForMaskedLM.from_pretrained(
    "biohub/ESMC-300M",
    device=device,
).eval()

tokenizer = EsmcTokenizer()

ESMC: Transformer Engine is not installed; falling back to pure-PyTorch LayerNorm+Linear / LayerNorm+MLP. Outputs will differ numerically - measured on the unnormalized residual stream (before the final LayerNorm), ~O(10) max-diff in fp32 and ~O(100) in bf16; after the final LayerNorm these shrink to a few ULP and perplexity stays within rounding noise. Install with `pip install transformer-engine[pytorch]` to enable fused fp32-reduction LayerNorm.
ESMC: neither xformers nor flash-attn is installed; falling back to PyTorch ``F.scaled_dot_product_attention``. The attention reduction order in bf16 differs from a fused kernel by ~1 bf16 ULP per attention block; compounded across the 80-block stack this reaches ~O(100) max-diff on the unnormalized residual stream. Install an xformers build matching your PyTorch and CUDA versions for a fused attention kernel.
ESMC: flash-attn rotary kernel not installed; falling back to pure-PyTorch RoPE. For faster GPU inference run `pip install flash-attn

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [5]:
def get_protein_embedding(amino_acid_seq):

    inputs = tokenizer([amino_acid_seq], return_tensors="pt", padding=True)
    inputs = {name: value.to(model.device) for name, value in inputs.items()}

    with torch.inference_mode():
        result = model(**inputs, output_hidden_states=True)

    token_embeddings = result.hidden_states[-1]    
    residue_embeddings = token_embeddings[0, 1:-1] # exlude start and end token

    return residue_embeddings

In [6]:
#nes is the reference relaxase used to identify sDBM by cosine blockwise matching

nes_seq = """
MAMYHFQNKFVSKANGQSATAKSAYNSASRIKDFKENEFKDYSNKQCDYSEILLPNNADD
KFKDREYLWNKVHDVENRKNSQVAREIIIGLPNEFDPNSNIELAKEFAESLSNEGMIVDL
NIHKINEENPHAHLLCTLRGLDKNNEFEPKRKGNDYIRDWNTKEKHNEWRKRWENVQNKH
LEKNGFSVRVSADSYKNQNIDLEPTKKEGWKARKFEDETGKKSSISKHNESIKKRNQQKI
DKMFNEVDDMKSHKLNAFSYMNKSDSTTLKNMAKDLKIYVTPINMYKENERLYDLKQKTS
LITDDEDRLNKIEDIEDRQKKLESINEVFEKQAGIFFDKNYPDQSLNYSDDEKSLSHMHY
INDRDVLPANNELEDIVKEKRIKEAQISLNTVLGNRDISLESIEKESNFFADKLSNILEK
NNLSFDDVLENKHEGMEDSLKIDYYTNKLEVFRNAENILEDYYDVQIKELFTDDEDYKAF
NEVTDIKEKQQLIDFKTYHGTENTIEMLETGNFIPKYSDEDRKYITEQVKLLQEKEFKPN
KNQHDKFVFGAIQKKLLSEYDFDYSDNNDLKHLYQESNEVGDEISKDNIEEFYEGNDILV
DKETYNNYNRKQQAYGLINAGLDSMIFNFNEIFRERMPKYINHQYKGKNHSKQRHELKNK
RGMHL
"""

nes_embed = get_protein_embedding(nes_seq)

nes_sDBM = nes_embed[4:13, :]

In [7]:
# nobaseL -> no of bases before nic site, nobaseR -> no of bases after nic site
def get_nic_seq(row, nobaseL = 7, nobaseR = 2):
    nic_site = row["nic site"]

    # assumes consecutive pos e.g. 4..5 -> always true for dataset
    nic_pos = int(nic_site.split(".")[0])

    oriT_seq = row["oriTDB_id"]

    # account for the fact that indexing in nic_pos begins at 1 (not 0)
    nic_site_start = nic_pos - nobaseL
    nic_site_end = nic_pos + nobaseR # excluded 

    if nic_pos < nobaseL or len(oriT_seq) <= (nic_pos + nobaseR - 1):
        return "NaN"

    return oriT_seq[nic_site_start: nic_site_end]


def query_sDBM(row, known_sDBM = nes_sDBM):

    protein_embed = get_protein_embedding(row["relaxase_sequence"])

    length, _ = protein_embed.shape

    max_mean = -1
    next_max_mean = -1

    start_seq = 0
    end_seq = 0 

    next_start = 0
    next_end = 0 

    for i in range(0, 30):
        match_block = protein_embed[i:i+9, :]

        sim = F.cosine_similarity(match_block, known_sDBM, dim = 1)

        mean = torch.mean(sim)

        if mean > max_mean:
            start_seq = i + 1
            end_seq = i + 9
            next_max_mean = max_mean
            max_mean = mean
        elif mean > next_max_mean:
            next_max_mean = mean
            next_start = i + 1
            next_end = i + 9

    perc = (max_mean - next_max_mean) * 100

    sDBM = row["relaxase_sequence"][start_seq : end_seq + 1]

    return sDBM, (start_seq, end_seq), float(perc)

In [26]:
filtered_df[["sDBM", "sDBM_pos", "sDBM_sim_gap"]] = (
    filtered_df.apply(query_sDBM, axis=1, result_type="expand")
)

filtered_df['oriT_nic_seq'] = filtered_df.apply(get_nic_seq, axis = 1)

filtered_df

,oriTDB_accession,oriTDB_id,nic site,relaxase_family,relaxase_sequence,host,sDBM,sDBM_pos,sDBM_sim_gap,oriT_nic_seq
0,100001,GAATAAGGGACAGTGAAGAAGGAACACCCGCTCGCGGGTGGGCCTA...,60..61,TraI,MIAKHVPMRSIKKSDFAELVKYITDEQGKTERLGHVRVTNCEANTL...,Pseudomonas aeruginosa,HVPMRSIKK,"(4, 12)",3.985026,TATCCTGCC
1,100002,CCGGCCAGCCTCGCAGAGCAGGATTCCCGTTGAGCACCGCCAGGTG...,107..108,TraI,MIAKHVPMRSIKKSDFAELVKYITDEQGKTERLGHVRVTNCEANTL...,Klebsiella aerogenes K8841,HVPMRSIKK,"(4, 12)",3.985026,TATCCTGCC
2,100003,CCAGTTTCTCGAAGAGAAACCGGTAAGTGCGCCCTCCC,31..32,MobA,MAIYHLTAKTGSRSGGQSARAKADYIQREGKYARDMDEVLHAESGH...,Escherichia coli,HLTAKTGSR,"(4, 12)",16.299099,AAGTGCGCC
3,100004,TTGCCCTATCCTGCATCGC,13..14,TaxC,MGVFVEKDYRVKRKSSENGRKSAFAHKVKNGGENYKKNVREKINRK...,E. coli,VEKDYRVKR,"(4, 12)",0.642514,TATCCTGCA
4,100005,TTGCCCTATCCTGCATCGC,13..14,TaxC,MGVFVEKDYRVKRKSSENGRKSAFAHKVKNGGENYKKNVREKINRK...,E. coli,VEKDYRVKR,"(4, 12)",0.642514,TATCCTGCA
...,...,...,...,...,...,...,...,...,...,...
5174,105175,AAATCTGCAAAATTTTAATTTTGCGTAGTGTGTGGTGATTTTGTGG...,33..34,traI_HV253_RS27590,MLSFSQVKSAGSAGNYYTEKDNYYVIGSMEERWQGKGAELLGLEGK...,Klebsiella pneumoniae strain RHBSTW-00636,SFSQVKSAG,"(2, 10)",1.182017,AGTGTGTGG
5179,105180,AATCTGCAAAATTTTAATTTTGCGTGGGGTGTGGTGATTTTGTGGTGAG,32..33,traI_HV295_RS23385,MMSIGSVKSAGKAGDYYTKEDNYYVIGSMDERWQGKGAEALGLEGK...,Enterobacter hormaechei strain RHBSTW-00844,SIGSVKSAG,"(2, 10)",2.994657,GGGGTGTGG
5181,110001,GGATTTAGGTTTTTTTTTAATCGCTTCACATTTCGTTAGCATGCGC...,78..79,MobH,MMMNFRALYLCIKRILGIFSSQENDATSVMIEDISSLSPFAQILGD...,Enterobacter roggenkampii strain 2017-45-51-04-01,NFRALYLCI,"(3, 11)",0.067639,AGTTTGGTT
5186,110006,GGATTTAGGTTTTTTTTTAATCGCTTCACATTTCGTTAGCATGCGA...,77..78,MobH,MMMNFRALYLCIKRILGIFSSQENDATSVMIEDISSLSPFAQILGD...,Escherichia coli strain W444,NFRALYLCI,"(3, 11)",0.064954,AGTTTGGTT


In [ ]:
# drop duplicate oriT-sDBM pairs (quite a lot due to data from different strains)
filtered_df = filtered_df.drop_duplicates(subset=["sDBM", "oriT_nic_seq"], keep='first')


In [11]:
filtered_df

,oriTDB_accession,oriTDB_id,nic site,relaxase_family,relaxase_sequence,host,sDBM,sDBM_pos,sDBM_sim_gap,oriT_nic_seq
0,100001,GAATAAGGGACAGTGAAGAAGGAACACCCGCTCGCGGGTGGGCCTA...,60..61,TraI,MIAKHVPMRSIKKSDFAELVKYITDEQGKTERLGHVRVTNCEANTL...,Pseudomonas aeruginosa,HVPMRSIKK,"(4, 12)",3.985026,TATCCTGCC
2,100003,CCAGTTTCTCGAAGAGAAACCGGTAAGTGCGCCCTCCC,31..32,MobA,MAIYHLTAKTGSRSGGQSARAKADYIQREGKYARDMDEVLHAESGH...,Escherichia coli,HLTAKTGSR,"(4, 12)",16.299099,AAGTGCGCC
3,100004,TTGCCCTATCCTGCATCGC,13..14,TaxC,MGVFVEKDYRVKRKSSENGRKSAFAHKVKNGGENYKKNVREKINRK...,E. coli,VEKDYRVKR,"(4, 12)",0.642514,TATCCTGCA
5,100006,ATACGAAGTAACGAAGTTACTGCGTATAAGTGCGCCCT,34..35,TraA,MTIAKRENGKRSLIAMASYRSGEKLYSELYEKTNLYNHRTVKPEAF...,Streptococcus agalactiae B-49813/75,IAKRENGKR,"(2, 10)",0.492692,AAGTGCGCC
6,100007,GGATTATCAATAGTCACATGTGAATCGGCCATTTATTTTATTAGAG...,305..306,TraI,MLSISSIKGDAGYYSHEDNYYASGSLDSRWMGEGAEKLGLKGEVAS...,Salmonella typhi,SISSIKGDA,"(2, 10)",2.794078,CGGGTGTGG
...,...,...,...,...,...,...,...,...,...,...
4257,104258,TCTGCCGATACTTTAAATATGAGTGTGCCTAATTTCGTTAAGAAAA...,66..67,Rep_trans_F1621_RS12210_SESURV_p2,MYSNSRLSGQTLGKSQPELSFDAMTIVGNLNKNNAQKLSEFMSVEP...,Staphylococcus epidermidis strain SESURV_p2_0614,TIVGNLNKN,"(24, 32)",0.684696,TCGATTGGT
4326,104327,CGACACAATCCAAAGGGGATAAAAGGGGAAAGTGAAACTTCCCCCT...,104..105,MobV,MAFMVARTEKRKVGSLAGYQNHVDRKTENHSNKEIDKSKTFLNYDL...,Lactococcus formosensis strain 122061,VARTEKRKV,"(4, 12)",4.477817,TAGCTTGCA
4367,104368,TCAGCTGAAACTTTAAATATGAGTGTGCCTACTTTCGTTAAGAAAA...,66..67,Rep_trans_PUW68_RS12235_pShVSI35,MSTENYSNYLQNKDLDNFSKTGYSNSRLDAHTVCISKPKISFDAMT...,Staphylococcus haemolyticus strain VSI35,YLQNKDLDN,"(8, 16)",0.109631,TCGATTGGT
4670,104671,AAAGAGCGATCTCGTCATCGGGGATTAGATTTCAAGGTGAAATACA...,111..112,MobP2,MDAPGVVLVSKYVSGKSTKFSKYVNYINRDEAVRTEKFQTYNVNKL...,Bacillus velezensis strain DSYZ,VLVSKYVSG,"(6, 14)",7.595348,TTTATGGTT


In [10]:
filtered_df.to_csv('../data/processed/cleaned_datasets/relaxase_oriT_dataset.csv', index=False)